# 00 — Setup & Data Download
**Project:** ViT Reliability & Explainability Under Medical Distribution Shift
**Author:** Sosna Worku

---
### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Have your Kaggle key ready (kaggle.com → Settings → API → Create New Token)

### Run this notebook only ONCE — data stays in Google Drive permanently.

## Step 1 — Clone GitHub repo

In [ ]:
import os, sys

REPO      = 'vit-medical-shift'
REPO_PATH = f'/content/{REPO}'
GITHUB    = 'https://github.com/sossyh/vit-medical-shift.git'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull origin main')
else:
    print('Cloning repo...')
    os.system(f'git clone {GITHUB} {REPO_PATH}')

sys.path.insert(0, REPO_PATH)
print('Done! Repo ready at', REPO_PATH)

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## Step 3 — Verify GPU

In [ ]:
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name      :', torch.cuda.get_device_name(0))
    print('GPU memory    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Step 4 — Install packages

In [ ]:
!pip install -q timm torchmetrics grad-cam einops pyyaml kaggle
print('All packages installed!')

## Step 5 — Set up Kaggle credentials
1. Go to kaggle.com → profile picture → Settings → API → Create New Token
2. Copy only the key string (the part after KGAT_)
3. Paste it below

In [ ]:
import os, json

# ── Fill in your key below ─────────────────────────────────
KAGGLE_USERNAME = 'sosnaworku'
KAGGLE_KEY      = 'paste_your_key_here'   # only the part after KGAT_
# ───────────────────────────────────────────────────────────

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials saved!')

# Test — should show list of files
!kaggle datasets files nih-chest-xrays/data

## Step 6 — Download & Extract NIH ChestX-ray14
Downloads ~7GB (2 image zips = ~18k images).
Takes 20-30 minutes — **do not close Colab.**

In [ ]:
import os, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/data/nih'
IMG_DIR    = f'{DRIVE_ROOT}/images'
os.makedirs(IMG_DIR, exist_ok=True)

# Download CSVs first
print('Downloading CSVs...')
for f in ['Data_Entry_2017.csv', 'BBox_List_2017.csv',
          'train_val_list.txt', 'test_list.txt']:
    dest = os.path.join(DRIVE_ROOT, f)
    if os.path.exists(dest):
        print(f'  {f} already exists, skipping.')
        continue
    subprocess.run(['kaggle', 'datasets', 'download',
                    '-d', 'nih-chest-xrays/data',
                    '-f', f, '--path', DRIVE_ROOT])
    zip_path = dest + '.zip'
    if os.path.exists(zip_path):
        subprocess.run(['unzip', '-q', '-o', zip_path, '-d', DRIVE_ROOT])
        os.remove(zip_path)
    print(f'  {f} done.')

# Download and extract image zips
for i in range(1, 3):  # change range(1,3) to range(1,5) for 4 zips later
    zip_name = f'images_{i:03d}.tar.gz'
    zip_path = f'{DRIVE_ROOT}/{zip_name}'

    # Download
    if not os.path.exists(zip_path):
        print(f'Downloading {zip_name} (~3.5GB)...')
        subprocess.run(['kaggle', 'datasets', 'download',
                        '-d', 'nih-chest-xrays/data',
                        '-f', zip_name, '--path', DRIVE_ROOT])
    else:
        print(f'{zip_name} already downloaded.')

    # Extract — strip 2 components removes images_001/images/ prefix
    print(f'Extracting {zip_name}...')
    result = subprocess.run([
        'tar', '-xzf', zip_path,
        '-C', IMG_DIR,
        '--strip-components=2'
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print('  Error:', result.stderr[:300])
    else:
        print('  Extracted OK!')

    # Remove zip to save Drive space
    if os.path.exists(zip_path):
        os.remove(zip_path)
        print(f'  Removed {zip_name} to save space.')

    n = len(os.listdir(IMG_DIR))
    print(f'  Images so far: {n:,}')

print(f'\nFinal image count: {len(os.listdir(IMG_DIR)):,}')
print('Sample files:', os.listdir(IMG_DIR)[:5])

## Step 7 — Verify everything

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/data/nih'
IMG_DIR    = f'{DRIVE_ROOT}/images'

checks = {
    'Data_Entry_2017.csv': os.path.join(DRIVE_ROOT, 'Data_Entry_2017.csv'),
    'BBox_List_2017.csv' : os.path.join(DRIVE_ROOT, 'BBox_List_2017.csv'),
    'images/ folder'     : IMG_DIR,
}

all_good = True
for name, path in checks.items():
    exists = os.path.exists(path)
    print(f'  {chr(10003) if exists else chr(10007)} {name}')
    if not exists:
        all_good = False

n = len(os.listdir(IMG_DIR)) if os.path.exists(IMG_DIR) else 0
print(f'  {chr(10003) if n > 0 else chr(10007)} {n:,} images found')
if n == 0:
    all_good = False

print()
if all_good:
    print('All good — ready to train!')
else:
    print('Something missing — check Step 6.')

## Done!
Data is permanently saved in Google Drive — never run this notebook again.

**Next step:** Open `01_data_exploration.ipynb`